In [ ]:
# Step 1: Data Cleaning & Preprocessing
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Load the dataset
df = pd.read_csv('Ride_Sharing.xlsx - Ride_Sharing.csv')

# Clean columns strictly following the syllabus
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Check for missing values
print("Missing Values Percentage:\n", df.isna().mean() * 100)

# Handle nulls
df = df.fillna(0)

# Convert dates to datetime objects
df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
df['ended_at'] = pd.to_datetime(df['ended_at'], errors='coerce')


In [ ]:
# Step 2: Feature Engineering & Groupby Analysis
# Calculate ride duration in minutes
df['duration_mins'] = (df['ended_at'] - df['started_at']).dt.total_seconds() / 60

# Extract the day of the week using the new syllabus method
df['day_of_week'] = df['started_at'].dt.day_name()

# Groupby analysis: Top 5 stations with highest total ride durations
top_stations = df.groupby('start_station_name')[['duration_mins']].sum().sort_values(by='duration_mins', ascending=False).head()
print("Top 5 Start Stations by Total Duration:\n", top_stations)


In [ ]:
# Step 3: Syllabus-Compliant Visualizations

# 1. Count of member vs casual riders
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='member_casual')
plt.title("Distribution of Rider Types")
plt.show()

# 2. Average ride duration by day of the week (strictly without estimator=sum or xticks)
plt.figure(figsize=(10, 4))
sns.barplot(data=df, x='day_of_week', y='duration_mins')
plt.title("Average Ride Duration by Day of Week")
plt.show()

# 3. Correlation Heatmap before Machine Learning
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap of Ride Sharing Data")
plt.show()


In [ ]:
# Step 4: Predictive Modeling - Logistic Regression
# We want to predict if a rider is 'member' or 'casual', so we use Logistic Regression

le = LabelEncoder()
# Encode categorical features
df['rideable_type_n'] = le.fit_transform(df['rideable_type'].astype(str))
df['day_of_week_n'] = le.fit_transform(df['day_of_week'].astype(str))

# Encode target variable
df['member_casual_n'] = le.fit_transform(df['member_casual'].astype(str))

# Define X (Inputs) and y (Output)
X = df[['rideable_type_n', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'duration_mins', 'day_of_week_n']]
y = df['member_casual_n']

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model (max_iter=10000 as required)
model = LogisticRegression(max_iter=10000)
model.fit(X_train, y_train)

# Score the model
print("Logistic Regression Model Score:", model.score(X_test, y_test))


In [ ]:
# Step 5: Model Prediction
# Dummy array corresponding to X: [rideable_type_n, start_lat, start_lng, end_lat, end_lng, duration_mins, day_of_week_n]
prediction = model.predict([[1, 41.9, -87.6, 41.9, -87.6, 15.5, 2]])
print("Prediction for new data (0 or 1):", prediction)
